# Атрибуты и признак общего бренда

Извлечённые характеристики сохраняются в `pairs_attrs.parquet`, кандидаты — в `brand_candidates.parquet`, признак пары — в `brand_features.parquet`. Сборка общей таблицы: `Feature_assembly.ipynb`.


## 1. Импорты


In [2]:
import pandas as pd
import numpy as np
import json
import re


## 2. Загрузка пар


In [4]:
pairs = pd.read_parquet('data/pairs.parquet', columns=['id1', 'id2', 'name_1', 'name_2', 'target', 'attributes_1', 'attributes_2'])

In [5]:
pairs.head()

,id1,id2,name_1,name_2,target,attributes_1,attributes_2
0,32,188978721454,очки с футляром-змейка fedrov мод. 1047 цвет 3...,фотохромные очки с футляром-змейка fedrov мод....,0.0,"{""диоптрии"":""- 2.00"",""цвет оправы"":""золотистый...","{""наличие флексов"":""да"",""затемнение"":""коричнев..."
1,34,197568543106,"зубная щетка r.o.c.s. black edition classic, ч...","зубная щетка r.o.c.s. red edition classic, сре...",0.0,"{""бренд"":""r.o.c.s."",""тип"":""зубная щетка"",""уров...","{""целевая аудитория"":""взрослая"",""бренд"":""r.o.c..."
2,37,661425040668,сандалии fre gamo,fre gamo / сандалии,0.0,"{""бренд в одежде и обуви"":""fre gamo"",""страна б...","{""цвет"":""желтый"",""страна производства"":""китай""..."
3,97,274877917548,acuvue астигматические линзы oasys for astigma...,acuvue / контактные линзы acuvue oasys for ast...,0.0,"{""ось (axis)"":""10"",""бренд"":""acuvue"",""цилиндр (...","{""состав"":""силикон-гидрогель"",""тип линз"":""тори..."
4,154,283467899529,"дворцовый пуэр (5 лет), 500 г",чайная линия ореховый пуэр (500 гр) вес: 500 гр,0.0,"{""бренд"":""weiserhouse"",""условия хранения"":""хра...","{""бренд"":""чайная линия"",""вид чая"":""пуэр"",""вкус..."


## 3. Очистка и извлечение атрибутов


In [6]:
no_article = ['нет', '-', '0', 'null', '_']
no_brand = ['нет бренда', 'не определен', 'noname', 'nobrand', 'без бренда', 'нет', 'нет бренд', '---','неизвестен', '.', 'туалетная вода', 'кеды', 'мягкая игрушка', 'производитель', 'оверсайз', 'no name']

def clean(value, bad_values, min_len=1):
    if value is None:
        return None
    value = value.lower().strip()
    if value == '' or value in bad_values or len(value) < min_len:
        return None
    return value



def parse_attributes(attribute):
    attribute = json.loads(attribute)
    brand_1 = attribute.get('бренд')
    brand_2 = attribute.get('бренд в одежде и обуви')
    article = attribute.get('артикул')
    brand_clean = clean(brand_1, no_brand, 2)

    if brand_clean is None:
        brand_clean = clean(brand_2, no_brand, 2)
    article_clean = clean(article, no_article, 3)
    partnumbers = ['артикул производителя', 'партномер (артикул производителя)', 'партномер']

    for partnumber in partnumbers:
        clean_partnumber= clean(attribute.get(partnumber), no_article, 3)
        if clean_partnumber is not None:
            break

    return brand_clean, article_clean, clean_partnumber


### Применение к обеим сторонам пары


In [7]:
parsed_1 = pairs['attributes_1'].apply(parse_attributes).tolist()
df = pd.DataFrame(parsed_1, columns=['brand', 'article', 'partnumber'])
pairs['brand_1'] = df['brand']
pairs['article_1'] = df['article']
pairs['partnumber_1'] = df['partnumber']

parsed_2 = pairs['attributes_2'].apply(parse_attributes).tolist()
df = pd.DataFrame(parsed_2, columns=['brand', 'article', 'partnumber'])
pairs['brand_2'] = df['brand']
pairs['article_2'] = df['article']
pairs['partnumber_2'] = df['partnumber']

In [8]:
pairs.head()

,id1,id2,name_1,name_2,target,attributes_1,attributes_2,brand_1,article_1,partnumber_1,brand_2,article_2,partnumber_2
0,32,188978721454,очки с футляром-змейка fedrov мод. 1047 цвет 3...,фотохромные очки с футляром-змейка fedrov мод....,0.0,"{""диоптрии"":""- 2.00"",""цвет оправы"":""золотистый...","{""наличие флексов"":""да"",""затемнение"":""коричнев...",None,None,None,fedrov,None,None
1,34,197568543106,"зубная щетка r.o.c.s. black edition classic, ч...","зубная щетка r.o.c.s. red edition classic, сре...",0.0,"{""бренд"":""r.o.c.s."",""тип"":""зубная щетка"",""уров...","{""целевая аудитория"":""взрослая"",""бренд"":""r.o.c...",r.o.c.s.,None,None,r.o.c.s.,None,None
2,37,661425040668,сандалии fre gamo,fre gamo / сандалии,0.0,"{""бренд в одежде и обуви"":""fre gamo"",""страна б...","{""цвет"":""желтый"",""страна производства"":""китай""...",fre gamo,None,None,None,None,None
3,97,274877917548,acuvue астигматические линзы oasys for astigma...,acuvue / контактные линзы acuvue oasys for ast...,0.0,"{""ось (axis)"":""10"",""бренд"":""acuvue"",""цилиндр (...","{""состав"":""силикон-гидрогель"",""тип линз"":""тори...",acuvue,None,None,None,None,None
4,154,283467899529,"дворцовый пуэр (5 лет), 500 г",чайная линия ореховый пуэр (500 гр) вес: 500 гр,0.0,"{""бренд"":""weiserhouse"",""условия хранения"":""хра...","{""бренд"":""чайная линия"",""вид чая"":""пуэр"",""вкус...",weiserhouse,None,None,чайная линия,None,None


### Проверка извлечённых данных


In [9]:
pairs = pairs[['id1', 'id2', 'target',
               'name_1', 'name_2',
               'brand_1', 'brand_2',
               'article_1', 'article_2',
               'partnumber_1', 'partnumber_2']]

In [10]:
print(pairs)

                 id1           id2  target  \
0                 32  188978721454     0.0   
1                 34  197568543106     0.0   
2                 37  661425040668     0.0   
3                 97  274877917548     0.0   
4                154  283467899529     0.0   
...              ...           ...     ...   
365649  850403683693  695784731688     0.0   
365650  850403683737  369367295238     0.0   
365651  850403683764  377957271291     1.0   
365652  850403683771  231928288442     0.0   
365653  850403683774  171798796415     0.0   

                                                   name_1  \
0       очки с футляром-змейка fedrov мод. 1047 цвет 3...   
1       зубная щетка r.o.c.s. black edition classic, ч...   
2                                       сандалии fre gamo   
3       acuvue астигматические линзы oasys for astigma...   
4                           дворцовый пуэр (5 лет), 500 г   
...                                                   ...   
365649  alcon контак

In [11]:
print(pairs.notna().mean())

id1             1.000000
id2             1.000000
target          1.000000
name_1          1.000000
name_2          1.000000
brand_1         0.490940
brand_2         0.490499
article_1       0.132738
article_2       0.133993
partnumber_1    0.093616
partnumber_2    0.093744
dtype: float64


## 4. Словарь брендов и товары без бренда


In [12]:
pd.set_option('display.max_colwidth', None)

brands_1 = pairs[['id1', 'brand_1', 'name_1']].rename(
    columns={'id1': 'id', 'brand_1': 'brand', 'name_1': 'name'}
)
brands_2 = pairs[['id2', 'brand_2', 'name_2']].rename(
    columns={'id2': 'id', 'brand_2': 'brand', 'name_2': 'name'}
)

brands = pd.concat([brands_1, brands_2], ignore_index=True)
brands = brands.drop_duplicates(subset='id')
without_brands = brands[brands['brand'].isna()]
brands = brands.dropna()

brand_counts = brands['brand'].value_counts()

brands_one = brand_counts[brand_counts == 1]

brands[brands['brand'].isin(brands_one.index)].sample(20)


,id,brand,name
658840,206158498722,viandar,"кофе в зернах 1 кг, зерновой кофе, тоффи, арабика 50% робуста 50%, viandar"
42150,94489360235,gentle monster,gentle monster очки солнцезащитные
405399,266288117646,brandart,футболка brandart
697802,335007546002,скрипка brahner bv412m 1/4,скрипка brahner bv412m 1/4 brahner bv412m 1/4 скрипка комплект кейс + смычок
37547,85899388131,siberiansun,шапка siberiansun премиальная линия
247557,575525752853,cl home,cl home плед полуторный 150х200 cleo parma велсофт бордовый
12653,25769852497,вудмакс,"вешалка в прихожую вешалка_универсальный, 40х8х4 см"
263191,618475303791,зоозащитники,"поводок брезентовый для собак, 5 метров, с усиленным карабином"
159928,369367353994,теплофон,"обогреватель инфракрасный электрический теплофон 700 вт для дома, для дачи"
201012,472446424664,baser,не определен поводок-рулетка 5 метров для животных


## 5. Поиск кандидатов в названиях


In [13]:
brands_sorted = sorted(brand_counts.index, key=len, reverse=True)
brands_pattern = '|'.join(re.escape(brand) for brand in brands_sorted)
brand_regex = re.compile(r'(?<!\w)(?:' + brands_pattern + r')(?!\w)')

def find_brand(name):
    found = brand_regex.findall(name.lower())
    return list(dict.fromkeys(found))




### Применение поиска и сохранение кандидатов


In [14]:
sample = without_brands.copy()
sample['brand_candidates'] = sample['name'].apply(find_brand)


In [20]:
sample['brand_candidates'].str.len().value_counts()


brand_candidates
1     188058
0      71724
2      71213
3      21924
4       5449
5       1306
6        593
7        197
8         30
9          9
10         2
Name: count, dtype: int64

In [17]:
sample.to_parquet('data/brand_candidates.parquet', index=False)

## 6. Присоединение кандидатов к парам


In [24]:
pairs_brands = pairs.merge(sample[['id', 'brand_candidates']], how='left', left_on='id1', right_on='id')

pairs_brands = pairs_brands.rename(
    columns={'brand_candidates': 'brand_candidates_1'}
)

pairs_brands = pairs_brands.drop(columns='id')

pairs_brands = pairs_brands.merge(sample[['id', 'brand_candidates']], how='left', left_on='id2', right_on='id')

pairs_brands = pairs_brands.rename(
    columns={'brand_candidates': 'brand_candidates_2'}
)

pairs_brands = pairs_brands.drop(columns='id')


## 7. Выбор источника бренда


In [ ]:
def get_brand_options(brand, candidates):
    if pd.notna(brand):
        return [brand]
    return candidates


In [29]:
pairs_brands['brands_1'] = pairs_brands.apply(
    lambda row: get_brand_options(row['brand_1'], row['brand_candidates_1']),
    axis=1
)

pairs_brands['brands_2'] = pairs_brands.apply(
    lambda row: get_brand_options(row['brand_2'], row['brand_candidates_2']),
    axis=1
)

## 8. Признак общего бренда


In [ ]:
def has_common_brand(brands_1, brands_2):
    if not brands_1 or not brands_2:
        return None
    else:
        common = set(brands_1) & set(brands_2)
        return int(bool(common))


In [38]:
pairs_brands['brand_common'] = pairs_brands.apply(
    lambda row: has_common_brand(row['brands_1'], row['brands_2']),
    axis=1
)





In [39]:
pairs_brands['brand_common'].value_counts(dropna=False)

brand_common
1.0    218550
0.0     87179
NaN     59925
Name: count, dtype: int64

In [40]:
pairs_brands

,id1,id2,target,name_1,name_2,brand_1,brand_2,article_1,article_2,partnumber_1,partnumber_2,brand_candidates_1,brand_candidates_2,brands_1,brands_2,brand_common
0,32,188978721454,0.0,очки с футляром-змейка fedrov мод. 1047 цвет 3 с линзами romeo 1.56 hmc/emi -2.00 рц 62-64,"фотохромные очки с футляром-змейка fedrov мод. 1047 цвет 3 с линзами romeo 1.56 fast photocolor brown, hmc+ -3.00 рц 64-66",None,fedrov,None,None,None,None,"[fedrov, romeo, emi]",NaN,"[fedrov, romeo, emi]",[fedrov],1.0
1,34,197568543106,0.0,"зубная щетка r.o.c.s. black edition classic, черная/красная","зубная щетка r.o.c.s. red edition classic, средняя",r.o.c.s.,r.o.c.s.,None,None,None,None,NaN,NaN,[r.o.c.s.],[r.o.c.s.],1.0
2,37,661425040668,0.0,сандалии fre gamo,fre gamo / сандалии,fre gamo,None,None,None,None,None,NaN,[fre gamo],[fre gamo],[fre gamo],1.0
3,97,274877917548,0.0,"acuvue астигматические линзы oasys for astigmatism,ось:10_-0.75, 6 шт., -0.75 / 8.6/-0.75/10, 2 недели",acuvue / контактные линзы acuvue oasys for astigmatism (6 линз) r 8.6 d +4.75 cyl -0.75 ax 10,acuvue,None,None,None,None,None,NaN,"[acuvue, acuvue oasys]",[acuvue],"[acuvue, acuvue oasys]",1.0
4,154,283467899529,0.0,"дворцовый пуэр (5 лет), 500 г",чайная линия ореховый пуэр (500 гр) вес: 500 гр,weiserhouse,чайная линия,None,None,None,None,NaN,NaN,[weiserhouse],[чайная линия],0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
365649,850403683693,695784731688,0.0,"alcon контактные линзы air optix night & day aqua, 3 шт. / 8.6 / 1 месяц, ежемесячные / +02.50","контактные линзы alcon air optix night & day aqua, 3 шт., r 8,6, d -9",alcon,None,None,None,None,None,NaN,"[alcon, air optix, aqua]",[alcon],"[alcon, air optix, aqua]",1.0
365650,850403683737,369367295238,0.0,shakmy / приправа для мяса 1 кг.,"приправа для мяса, 1кг",None,carp box,None,None,None,None,[],NaN,[],[carp box],NaN
365651,850403683764,377957271291,1.0,"3d art.игрушка-раскраска ""снежный барс""","lori / набор для творчества раскраска игрушка "" 93335",lori,None,None,None,None,None,NaN,"[lori, игрушка]",[lori],"[lori, игрушка]",1.0
365652,850403683771,231928288442,0.0,"грузовик радиоуправляемый самосвал, работает от батареек,","грузовик радиоуправляемый «самосвал», работает от батареек,",None,None,None,None,None,None,[],[],[],[],NaN


## 9. Сохранение результатов


In [41]:
pairs_brands[['id1', 'id2', 'target', 'brand_common']].to_parquet(
    'data/brand_features.parquet',
    index=False
)


In [42]:
pairs.to_parquet('data/pairs_attrs.parquet')

### Вывод

Создан признак пары `brand_common`. Для каждого товара используется очищенный бренд из характеристик, а при его отсутствии — кандидаты, найденные в названии по словарю брендов.

Признак равен:

- **1** — есть хотя бы один общий бренд;
- **0** — информация есть у обоих товаров, но общих брендов нет;
- **NaN** — хотя бы у одного товара нет информации о бренде.


In [3]:
pairs_attrs = pd.read_parquet('data/pairs_attrs.parquet')


In [13]:
def compare_attribute(value_1, value_2):
    if pd.isna(value_1) or pd.isna(value_2):
        return None
    if value_1 == value_2:
        return 1
    return 0

pairs_attrs['article_equal'] = pairs_attrs.apply(
    lambda row: compare_attribute(row['article_1'], row['article_2']),
    axis=1
)

pairs_attrs['partnumber_equal'] = pairs_attrs.apply(
    lambda row: compare_attribute(row['partnumber_1'], row['partnumber_2']),
    axis=1
)

pairs_attrs['brand_equal'] = pairs_attrs.apply(
    lambda row: compare_attribute(row['brand_1'], row['brand_2']),
    axis=1
)


In [9]:
pairs_attrs["article_equal"].value_counts(dropna=False)


article_equal
NaN    363647
0.0      1981
1.0        26
Name: count, dtype: int64

In [10]:
pairs_attrs["partnumber_equal"].value_counts(dropna=False)

partnumber_equal
NaN    355834
0.0      8906
1.0       914
Name: count, dtype: int64

In [14]:
pairs_attrs['brand_equal'].value_counts(dropna=False)

brand_equal
NaN    322168
1.0     29380
0.0     14106
Name: count, dtype: int64

In [15]:
pairs_attrs.to_parquet('data/pairs_attrs.parquet', index=False)